In [ ]:
# ── polished status box (the ONLY thing this cell displays) ────
_BOX_CSS = ('<style>'
 '.wk-root{max-width:680px;margin:14px auto;background:#15151a;border:1px solid #26262e;border-radius:10px;overflow:hidden;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:#e8e8ea;}'
 '.wk-hdr{display:flex;align-items:center;gap:8px;padding:10px 14px;border-bottom:1px solid #26262e;background:#17171c;}'
 '.wk-title{font-size:13px;font-weight:600;letter-spacing:.02em;}'
 '.wk-status{margin-left:auto;display:flex;align-items:center;gap:6px;font-size:10px;letter-spacing:.12em;text-transform:uppercase;color:#9a9aa3;}'
 '.wk-dot{width:7px;height:7px;border-radius:50%;flex-shrink:0;}'
 '.wk-body{padding:12px 14px;}'
 '.wk-msg{font-size:12px;color:#c9c9d1;margin-bottom:6px;line-height:1.5;overflow-wrap:anywhere;}'
 '.wk-row{display:flex;justify-content:space-between;gap:12px;padding:5px 0;border-top:1px solid #202028;font-size:12px;}'
 '.wk-row:first-child{border-top:none;}'
 '.wk-k{color:#8a8a95;}'
 '.wk-v{color:#e8e8ea;text-align:right;word-break:break-all;}'
 '.wk-v.bad{color:#ff9d9d;}'
 '.wk-v.ok{color:#7ee2a0;}'
 '.wk-btn{display:inline-block;margin:4px 0 8px;padding:9px 22px;border-radius:6px;background:#e8e8ea;color:#101014 !important;text-decoration:none;font-size:12px;font-weight:600;}'
 '.wk-mono{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:11px;color:#8a8a95;word-break:break-all;}'
 '.wk-foot{padding:8px 14px;border-top:1px solid #26262e;background:#17171c;font-size:10px;color:#8a8a95;overflow-wrap:anywhere;}'
 '</style>')
def _wk_dot(kind, text):
    c = {'run': '#e8c96a', 'ok': '#7ee2a0', 'bad': '#ff9d9d', 'idle': '#8a8a95'}[kind]
    return "<span class='wk-dot' style='background:%s'></span><span>%s</span>" % (c, _html.escape(str(text)))
def _box(title, kind, status, msg, rows=(), url=None, foot=''):
    items = []
    for k, v in rows:
        v = str(v)
        cls = 'bad' if 'FAIL' in v else ('ok' if v in ('done', 'ok', 'READY') else '')
        items.append("<div class='wk-row'><span class='wk-k'>%s</span><span class='wk-v %s'>%s</span></div>" % (_html.escape(str(k)), cls, _html.escape(v)))
    link = ''
    if url:
        u = _html.escape(str(url), quote=True)
        link = "<a class='wk-btn' href='%s' target='_blank'>OPEN STUDIO</a><div class='wk-mono'>%s</div>" % (u, u)
    body = ("<div class='wk-root'><div class='wk-hdr'><span class='wk-title'>%s</span>"
            "<span class='wk-status'>%s</span></div><div class='wk-body'><div class='wk-msg'>%s</div>%s%s</div>"
            "<div class='wk-foot'>%s</div></div>") % (_html.escape(str(title)), _wk_dot(kind, status), _html.escape(str(msg)), ''.join(items), link, _html.escape(str(foot)))
    return _BOX_CSS + body
import html as _html
from IPython.display import display, HTML

# @title 1 · Environment + /kaggle/tmp storage
# WeeLLM Studio · Kaggle · Cell 1 of 2 · Settings → Accelerator → GPU, then Run All.
# Downloads route to /kaggle/tmp (HF cache symlinked there). Only the box shows.
import os, sys, subprocess, time, shutil, warnings, contextlib, io, threading
warnings.simplefilter('ignore')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

_TITLE = 'WeeLLM Studio · Environment'
_results = []
_handle = display(HTML(_box(_TITLE, 'run', 'INSTALLING', 'Cell 1 of 2 — installing packages; big downloads take minutes.', foot='Kaggle · downloads → /kaggle/tmp')), display_id=True)
def _ref(kind, status, msg, foot=''):
    _handle.update(HTML(_box(_TITLE, kind, status, msg, _results, foot=foot)))

# route downloads to /kaggle/tmp and link the HF cache there (silent)
TMP = '/kaggle/tmp'
HF_HOME = os.path.join(TMP, 'huggingface')
os.makedirs(os.path.join(HF_HOME, 'hub'), exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
os.environ['HF_HUB_CACHE'] = os.path.join(HF_HOME, 'hub')
_default_cache = os.path.expanduser('~/.cache/huggingface')
os.makedirs(os.path.dirname(_default_cache), exist_ok=True)
if not os.path.exists(_default_cache):
    os.symlink(HF_HOME, _default_cache)
OUT_DIR = '/kaggle/working/weellm_studio'
os.makedirs(OUT_DIR, exist_ok=True)
os.environ['WEELLM_OUT_DIR'] = OUT_DIR
_results += [('HF cache', os.environ['HF_HUB_CACHE']), ('cache link', _default_cache), ('outputs', OUT_DIR)]
_ref('run', 'INSTALLING', 'Routing model downloads to /kaggle/tmp…')

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import torch
if not torch.cuda.is_available():
    _results.append(('gpu', 'FAILED — set Settings → Accelerator → GPU'))
    _ref('bad', 'FAILED', 'No GPU detected. Enable GPU and re-run this cell.')
    raise SystemExit('No GPU')
_results.append(('gpu', torch.cuda.get_device_name(0)))
_ref('run', 'INSTALLING', 'GPU present — installing packages…')

_parts = (torch.version.cuda or 'cpu').split('.')
_tag = 'cu' + _parts[0] + _parts[1] if _parts[0].isdigit() else 'cpu'
_IDX = 'https://download.pytorch.org/whl/' + _tag
STEPS = [
    ('Base tools (hf_transfer · gradio · peft)', [sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer', 'psutil', 'gradio>=5', 'peft>=0.10.0']),
    ('torchao upgrade', [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao', '--index-url', _IDX]),
    ('diffusers @ git', [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/diffusers.git']),
    ('WeeLLM-Enhanced @ git (fresh, no deps)', [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', 'git+https://github.com/GodL-x-SouL/WeeLLM-Enhanced.git']),
    ('WeeLLM deps (fill gaps)', [sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'safetensors', 'numpy', 'Pillow', 'tqdm', 'gguf', 'av', 'huggingface_hub']),
]
_HB = {'step': 'starting', 't0': time.time(), 'done': False}
def _beat():
    while not _HB['done']:
        time.sleep(30)
        try:
            _ref('run', 'INSTALLING', '%s — still working… %ds elapsed (do not interrupt)' % (_HB['step'], int(time.time() - _HB['t0'])), foot='keep this cell running')
        except Exception:
            pass
threading.Thread(target=_beat, daemon=True).start()
t0 = time.time()
ok_all = True
for name, cmd in STEPS:
    _HB['step'] = name
    _ref('run', 'INSTALLING', 'Running: %s …' % name)
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=1500)
    ok = (p.returncode == 0)
    ok_all = ok_all and ok
    _results.append((name, 'done' if ok else 'FAILED — ' + (p.stderr or '')[-200:]))
    _ref('run', 'INSTALLING', 'Installing…')
_HB['done'] = True

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        try:
            import gradio, diffusers, huggingface_hub, weellm
            from torchao import quantization as _tq
            assert hasattr(_tq, 'FqnToConfig'), 'torchao too old'
            import av
            _results.append(('torch / gradio / weellm', '%s / %s / %s' % (torch.__version__, gradio.__version__, weellm.__version__)))
            _results.append(('/kaggle/tmp free', '%.0f GB' % (shutil.disk_usage(TMP).free / 1e9)))
        except Exception as e:
            ok_all = False
            _results.append(('verify', 'FAILED — %s: %s' % (type(e).__name__, str(e)[-160:])))
if ok_all:
    _ref('ok', 'READY', 'Cell 1 done — run Cell 2 to open the Studio.', foot='setup %.0fs' % (time.time() - t0))
else:
    _ref('bad', 'FAILED', 'Something failed — read the rows above, fix, and re-run this cell.')


In [ ]:
# ── polished status box (the ONLY thing this cell displays) ────
_BOX_CSS = ('<style>'
 '.wk-root{max-width:680px;margin:14px auto;background:#15151a;border:1px solid #26262e;border-radius:10px;overflow:hidden;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:#e8e8ea;}'
 '.wk-hdr{display:flex;align-items:center;gap:8px;padding:10px 14px;border-bottom:1px solid #26262e;background:#17171c;}'
 '.wk-title{font-size:13px;font-weight:600;letter-spacing:.02em;}'
 '.wk-status{margin-left:auto;display:flex;align-items:center;gap:6px;font-size:10px;letter-spacing:.12em;text-transform:uppercase;color:#9a9aa3;}'
 '.wk-dot{width:7px;height:7px;border-radius:50%;flex-shrink:0;}'
 '.wk-body{padding:12px 14px;}'
 '.wk-msg{font-size:12px;color:#c9c9d1;margin-bottom:6px;line-height:1.5;overflow-wrap:anywhere;}'
 '.wk-row{display:flex;justify-content:space-between;gap:12px;padding:5px 0;border-top:1px solid #202028;font-size:12px;}'
 '.wk-row:first-child{border-top:none;}'
 '.wk-k{color:#8a8a95;}'
 '.wk-v{color:#e8e8ea;text-align:right;word-break:break-all;}'
 '.wk-v.bad{color:#ff9d9d;}'
 '.wk-v.ok{color:#7ee2a0;}'
 '.wk-btn{display:inline-block;margin:4px 0 8px;padding:9px 22px;border-radius:6px;background:#e8e8ea;color:#101014 !important;text-decoration:none;font-size:12px;font-weight:600;}'
 '.wk-mono{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:11px;color:#8a8a95;word-break:break-all;}'
 '.wk-foot{padding:8px 14px;border-top:1px solid #26262e;background:#17171c;font-size:10px;color:#8a8a95;overflow-wrap:anywhere;}'
 '</style>')
def _wk_dot(kind, text):
    c = {'run': '#e8c96a', 'ok': '#7ee2a0', 'bad': '#ff9d9d', 'idle': '#8a8a95'}[kind]
    return "<span class='wk-dot' style='background:%s'></span><span>%s</span>" % (c, _html.escape(str(text)))
def _box(title, kind, status, msg, rows=(), url=None, foot=''):
    items = []
    for k, v in rows:
        v = str(v)
        cls = 'bad' if 'FAIL' in v else ('ok' if v in ('done', 'ok', 'READY') else '')
        items.append("<div class='wk-row'><span class='wk-k'>%s</span><span class='wk-v %s'>%s</span></div>" % (_html.escape(str(k)), cls, _html.escape(v)))
    link = ''
    if url:
        u = _html.escape(str(url), quote=True)
        link = "<a class='wk-btn' href='%s' target='_blank'>OPEN STUDIO</a><div class='wk-mono'>%s</div>" % (u, u)
    body = ("<div class='wk-root'><div class='wk-hdr'><span class='wk-title'>%s</span>"
            "<span class='wk-status'>%s</span></div><div class='wk-body'><div class='wk-msg'>%s</div>%s%s</div>"
            "<div class='wk-foot'>%s</div></div>") % (_html.escape(str(title)), _wk_dot(kind, status), _html.escape(str(msg)), ''.join(items), link, _html.escape(str(foot)))
    return _BOX_CSS + body
import html as _html
from IPython.display import display, HTML

# @title 2 · Studio server (keep this cell running)
# WeeLLM Studio · Kaggle · Cell 2 of 2 · The UI lives in the separate GradioUI.py
# (not embedded here). This cell serves it as a share URL. Only the box shows;
# Gradio/tunnel noise is captured and the heartbeat refreshes the SAME box.
import os, sys, time, glob, shutil, urllib.request, contextlib, io
_TITLE = 'WeeLLM Studio'
_handle = display(HTML(_box(_TITLE, 'run', 'STARTING', 'Cell 2 of 2 — locating GradioUI.py and opening a share tunnel…', foot='Kaggle · keep this cell alive')), display_id=True)
def _ref(kind, status, msg, url=None, foot=''):
    _handle.update(HTML(_box(_TITLE, kind, status, msg, [], url=url, foot=foot)))

WORK = '/kaggle/working'
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
_capo, _cape = io.StringIO(), io.StringIO()
# Redirect for the WHOLE serving lifetime: Gradio/tunnel prints from background
# threads (e.g. "Running on public URL") must never leak outside the box.
with contextlib.redirect_stdout(_capo), contextlib.redirect_stderr(_cape):
    try:
        _target = os.path.join(WORK, 'GradioUI.py')
        _RAW = 'https://raw.githubusercontent.com/GodL-x-SouL/WeeLLM-Enhanced/main/GradioUI.py'
        # An explicitly uploaded copy wins (deliberate local edits), otherwise ALWAYS
        # re-download the latest studio: /kaggle/working/GradioUI.py survives between
        # runs in the same session, so reusing it silently pins you to a stale build.
        _uploaded = sorted(glob.glob('/kaggle/input/**/GradioUI.py', recursive=True))
        try:
            if _uploaded:
                shutil.copyfile(_uploaded[0], _target)
            else:
                urllib.request.urlretrieve(_RAW, _target + '.new')
                shutil.move(_target + '.new', _target)
        except Exception as _e:
            _ref('err', 'FAIL', 'Could not fetch GradioUI.py: %s' % _e)
            raise
        import GradioUI as studio
        _url = getattr(studio, '_URL', None)
        if not _url:
            _url = studio.serve()      # share=True on Kaggle; returns public URL
            studio._URL = _url
        _ref('ok', 'LIVE', 'The app is running — open the link below. Keep this cell and the session alive.', url=_url, foot='serving · models stream from /kaggle/tmp')
        import torch
        _t0, _beats = time.time(), 0
        while True:  # keep-alive: refresh the SAME box, never print new lines
            time.sleep(60)
            _beats += 1
            _up = int(time.time() - _t0)
            try:
                _vram = torch.cuda.memory_allocated() / 1e9
            except Exception:
                _vram = -1.0
            _ref('ok', 'LIVE', 'Serving — open the link below. Stop this cell to shut down.', url=_url,
                 foot='heartbeat #%d · uptime %dh %02dm · VRAM %.2f GB' % (_beats, _up // 3600, (_up % 3600) // 60, _vram))
    except KeyboardInterrupt:
        _ref('bad', 'STOPPED', 'Keep-alive interrupted — Studio stopped. Re-run this cell to restart.')
    except Exception as e:
        _ref('bad', 'ERROR', 'Launch failed: %s: %s' % (type(e).__name__, e), foot=(_cape.getvalue() or _capo.getvalue())[-300:])
